<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week7_Day5_Exercises_XP_Prompt_Engineering_Gold_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP Gold ? Prompt Engineering
Last Updated: October 7th, 2025
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Craft precise prompts for complex tasks.
- Debug and refine chain-of-thought (CoT) reasoning.
- Select prompt patterns for real-world applications and reduce hallucinations.
- Design chained prompts with conditional logic, bias mitigation, and simulated memory.

## What you'll build
- Corrected CoT prompts and answers.
- Domain-specific prompt patterns and multi-step pipelines.
- Fair role-based prompts and conversational agents with memory.

## Helper: Run a prompt
Uses `ollama run` if available (set `OLLAMA_MODEL` to override, default: `llama3`). If Ollama is missing or the call fails, the helper prints the prompt in dry-run mode instead of crashing.


In [11]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


## Exercise 1: Debug a Faulty Chain-of-Thought
Goal: Fix the CoT and final answer. Original prompt had math/logic issues for the pencil change problem.

### Tasks
- Find the reasoning mistake.
- Rewrite the CoT prompt with correct steps.
- Provide the correct answer.

In [12]:
# TODO: describe the mistake
cot_issue = 'The multiplication 6 * $0.75 was incorrectly calculated as $4.75 instead of $4.50, which also caused the subtraction from $5.00 to be wrong.'
cot_issue

'The multiplication 6 * $0.75 was incorrectly calculated as $4.75 instead of $4.50, which also caused the subtraction from $5.00 to be wrong.'

In [13]:
# TODO: rewrite the CoT prompt
fixed_cot_prompt = '''A shop sells pencils at $0.75 each. If Alice buys 6 pencils and pays with a $5 bill, how much change does she get? Let’s solve this step-by-step:
1. Calculate the total cost of the pencils: 6 * $0.75.
2. Subtract that total from the $5.00 bill to find the change.
3. State the final amount.'''
fixed_cot_prompt

'A shop sells pencils at $0.75 each. If Alice buys 6 pencils and pays with a $5 bill, how much change does she get? Let’s solve this step-by-step:\n1. Calculate the total cost of the pencils: 6 * $0.75.\n2. Subtract that total from the $5.00 bill to find the change.\n3. State the final amount.'

In [14]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


In [15]:
# TODO: final correct answer
# 6 * 0.75 = 4.50
# 5.00 - 4.50 = 0.50
correct_change = '$0.50'
correct_change

'$0.50'

## Exercise 2: Choose the Right Prompt Pattern
Goal: Pick and justify a prompt pattern for support ticket classification (Billing Issue, Technical Support, Account Access, Other).

### Tasks
- Choose a pattern (Zero-Shot, Few-Shot, IAP, LoT, etc.).
- Write a complete prompt using that pattern.
- Justify the choice (ambiguity, consistency, generalization).

In [16]:
# TODO: choose pattern and prompt
chosen_pattern = 'Few-Shot Prompting'
classification_prompt = '''You are a customer support assistant. Categorize the following customer message into one of these classes: Billing Issue, Technical Support, Account Access, Other.

Examples:
Message: "I was charged twice for my subscription this month."
Category: Billing Issue

Message: "I can't log in to my dashboard, it says my password is wrong."
Category: Account Access

Message: "The app keeps crashing whenever I try to upload a photo."
Category: Technical Support

Message: "Do you have a dark mode available?"
Category: Other

Message: "I need to reset my security questions."
Category: '''
justification = 'Few-Shot is best here because it defines the label space clearly and provides context through examples. This reduces ambiguity in how the model should categorize borderline cases and ensures the output format remains consistent for downstream processing.'
print(f"Pattern: {chosen_pattern}\n")
print(classification_prompt)

Pattern: Few-Shot Prompting

You are a customer support assistant. Categorize the following customer message into one of these classes: Billing Issue, Technical Support, Account Access, Other.

Examples:
Message: "I was charged twice for my subscription this month."
Category: Billing Issue

Message: "I can't log in to my dashboard, it says my password is wrong."
Category: Account Access

Message: "The app keeps crashing whenever I try to upload a photo."
Category: Technical Support

Message: "Do you have a dark mode available?"
Category: Other

Message: "I need to reset my security questions."
Category: 


In [17]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


## Exercise 3: Use AlignedCoT to Compare Reasoning Paths
Goal: Two reasoning paths + comparison for the flower pot cost.

### Tasks
- Build AlignedCoT with at least two distinct reasoning structures.
- Add a comparison step to select the consistent answer.

In [18]:
# Exercise 3: Aligned CoT
aligned_cot_prompt = '''Calculate the total cost for: 2 small pots ($2 each), 3 medium pots ($4 each), and 1 large pot ($6 each).

Path 1: Calculate the cost for each type individually and sum them.
Path 2: Sum the unit costs of all pots purchased and group by frequency.

Comparison: Compare the results of Path 1 and Path 2. If they match, provide the final total. If not, re-calculate.'''
aligned_cot_prompt

'Calculate the total cost for: 2 small pots ($2 each), 3 medium pots ($4 each), and 1 large pot ($6 each).\n\nPath 1: Calculate the cost for each type individually and sum them.\nPath 2: Sum the unit costs of all pots purchased and group by frequency.\n\nComparison: Compare the results of Path 1 and Path 2. If they match, provide the final total. If not, re-calculate.'

In [19]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


## Exercise 4: Design a Multi-Step Document Pipeline
Goal: Three-stage pipeline (domain detect -> contributions -> follow-up question) with conditional/context chaining.

### Tasks
- Draft prompt template for each stage.
- Note where to chain context or branch conditionally.

In [20]:
# Exercise 4: Multi-Step Pipeline
stage1_domain = 'Identify the primary academic domain (e.g., Biology, Physics, CS) of this abstract: [ABSTRACT TEXT]'
stage2_contrib = 'Given that this paper is in the domain of [DOMAIN], list the top 3 key technical contributions: [ABSTRACT TEXT]'
stage3_followup = 'Based on the contributions [CONTRIBUTIONS], propose a follow-up research question for future study.'
chaining_notes = 'Conditional logic is useful after Stage 1: if domain is CS, use a code-specific extraction prompt; if Biology, use a clinical-specific prompt.'
stage1_domain, stage2_contrib, stage3_followup

('Identify the primary academic domain (e.g., Biology, Physics, CS) of this abstract: [ABSTRACT TEXT]',
 'Given that this paper is in the domain of [DOMAIN], list the top 3 key technical contributions: [ABSTRACT TEXT]',
 'Based on the contributions [CONTRIBUTIONS], propose a follow-up research question for future study.')

## Exercise 5: Role Prompting to Reduce Bias
Goal: Fair career recommendations.

### Tasks
- Write a basic prompt (may be biased).
- Write a revised role-based prompt to reduce bias.
- Explain how the role prompt improves fairness.

In [21]:
# Exercise 5: Role Prompting
biased_prompt = 'The user is interested in helping people and is good at biology. Suggest a career path.'
debiased_prompt = 'You are an unbiased Career Counselor. Based on the user skills [Biology] and interests [Helping people], suggest 3 careers. Ensure suggestions are based strictly on qualifications and ignore gender or cultural stereotypes.'
fairness_explanation = 'The role prompt sets a professional boundary that prioritizes objective skill-matching over societal biases.'
debiased_prompt

'You are an unbiased Career Counselor. Based on the user skills [Biology] and interests [Helping people], suggest 3 careers. Ensure suggestions are based strictly on qualifications and ignore gender or cultural stereotypes.'

In [22]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None


## Exercise 6: Build a Conversational Agent with Context Memory
Goal: Add memory to a virtual health coach.

### Tasks
- Pick a memory technique (prior message passing, structured history, vector store retrieval).
- Show how you would structure past context.
- Write a prompt that injects that context for the next response.

In [23]:
# Exercise 6: Conversational Memory
memory_approach = 'Structured History Passing'
past_context = '''User Profile: Prefers keto diet, sleeps 6 hours, goal is weight loss.
Past Advice: Recommended increasing sleep to 7 hours and adding a 15-minute walk.'''
next_turn_prompt = f'''Context: {past_context}
User: I felt tired during my walk today. What should I change?
Coach: '''
next_turn_prompt

'Context: User Profile: Prefers keto diet, sleeps 6 hours, goal is weight loss.\nPast Advice: Recommended increasing sleep to 7 hours and adding a 15-minute walk.\nUser: I felt tired during my walk today. What should I change?\nCoach: '

In [24]:
import os, subprocess

def run_prompt(prompt: str, model: str | None = None, temperature: float = 0.7, max_new_tokens: int = 120):
    """Send a single-turn prompt via `ollama run`.

    - Set OLLAMA_MODEL env var or pass model to override.
    - Falls back to dry-run if ollama is unavailable."""
    model = model or os.environ.get('OLLAMA_MODEL', 'llama3')
    cmd = ['ollama', 'run', model]
    try:
        proc = subprocess.run(cmd, input=prompt.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        out = proc.stdout.decode('utf-8', errors='ignore')
        print(out)
        return out
    except FileNotFoundError:
        print('[dry-run] ollama not installed. Prompt to send:', prompt)
    except subprocess.CalledProcessError as e:
        err = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
        print('[dry-run] ollama call failed:', err)
    return None
